In [9]:
# Скелет Этапа 1

import numpy as np
import torch

import torch

class UniformQuantizer:
    def __init__(self, bits: int, symmetric: bool, granularity: str):
        self.bits = bits
        self.symmetric = symmetric
        self.granularity = granularity
        self.s = None
        self.z = None

    def compute_params(self, W: torch.Tensor):
        # Определяем границы сетки
        if self.symmetric:
            q_max = 2**(self.bits - 1) - 1
            q_min = -q_max
        else:
            q_min, q_max = 0, 2**self.bits - 1

        # Выбираем оси для редукции (сжатия)
        # Если per_tensor -> сжимаем всё до скаляра
        # Если per_channel -> сжимаем по строкам (dim=1)
        reduce_dims = (1,) if self.granularity == 'per_channel' else None

        if self.symmetric:
            # Считаем abs_max. Если dim=1, получим вектор [O, 1]
            if reduce_dims:
                abs_max = W.abs().amax(dim=reduce_dims, keepdim=True)
            else:
                abs_max = W.abs().max()
                
            s = abs_max / q_max
            z = torch.zeros_like(s)
        else:
            if reduce_dims:
                w_min = W.amin(dim=reduce_dims, keepdim=True)
                w_max = W.amax(dim=reduce_dims, keepdim=True)
            else:
                w_min, w_max = W.min(), W.max()
                
            s = (w_max - w_min) / (q_max - q_min)
            # Важно: ограничиваем s снизу, чтобы не делить на ноль
            s = torch.clamp(s, min=1e-8) 
            z = torch.round(q_min - w_min / s)
            
        return s, z, q_min, q_max

    def quantize(self, W: torch.Tensor) -> torch.Tensor:
        s, z, q_min, q_max = self.compute_params(W)
        self.s, self.z = s, z
        
        # Сама формула:
        W_q = torch.round(W / s + z)
        return torch.clamp(W_q, q_min, q_max) # Ограничиваем сеткой

    def dequantize(self, W_q: torch.Tensor) -> torch.Tensor:
        return (W_q - self.z) * self.s

    def quantization_error(self, W: torch.Tensor) -> float:
        # MSE(W, dequantize(quantize(W)))
        w_hat = self.dequantize(self.quantize(W))
        diff = torch.sum(torch.square(W - w_hat))
        return (diff / W.numel()).item()

In [10]:
W = torch.randn(768, 768)  # типичный размер attention-матрицы в BERT-base

for bits in [8, 4]:
    for sym in [True, False]:
        for gran in ['per_tensor', 'per_channel']:
            q = UniformQuantizer(bits, sym, gran)
            err = q.quantization_error(W)
            print(f"INT{bits}, {'sym' if sym else 'asym'}, {gran}: MSE={err:.6f}")

INT8, sym, per_tensor: MSE=0.000115
INT8, sym, per_channel: MSE=0.000059
INT8, asym, per_tensor: MSE=0.000113
INT8, asym, per_channel: MSE=0.000051
INT4, sym, per_tensor: MSE=0.037697
INT4, sym, per_channel: MSE=0.019236
INT4, asym, per_tensor: MSE=0.032557
INT4, asym, per_channel: MSE=0.014759


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", torch_dtype=torch.float32)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

In [12]:
tokenizer = AutoTokenizer.from_pretrained("facebook/opt-125m")

In [13]:
from torch import nn
import torch.nn.functional as F

class QuantizedLinear(nn.Module):
    def __init__(self, original_linear: nn.Linear, quantizer):
        super().__init__()
        
        W = original_linear.weight.data
        W_q_tensor = quantizer.quantize(W)
        
        # Сохраняем веса, масштаб и смещение как параметры текущего слоя
        self.W_q = nn.Parameter(W_q_tensor, requires_grad=False)
        self.scale = nn.Parameter(quantizer.s.clone(), requires_grad=False)
        self.zero_point = nn.Parameter(quantizer.z.clone(), requires_grad=False)
        self.bias = original_linear.bias

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Выполняем деквантование, используя собственные сохраненные параметры
        W_hat = (self.W_q - self.zero_point) * self.scale
        return F.linear(x, W_hat.to(x.dtype), self.bias)


import copy

def replace_linear_layers(model: nn.Module, quantizer: UniformQuantizer) -> nn.Module:
    """Рекурсивно заменяет все nn.Linear в модели."""
    for name, module in model.named_children():
        if isinstance(module, nn.Linear):
            # Передаем независимую копию квантизатора
            setattr(model, name, QuantizedLinear(module, copy.deepcopy(quantizer)))
        else:
            replace_linear_layers(module, quantizer)
    return model

In [14]:
quantized_model_ch = replace_linear_layers(model, UniformQuantizer(8, True, 'per_channel'))
quantized_model_tens = replace_linear_layers(model, UniformQuantizer(8, True, 'per_channel'))
# 'per_tensor',

In [15]:
import torch
from tqdm import tqdm
from datasets import load_dataset

def compute_perplexity(model, tokenizer, n_samples=200, seq_len=512):
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    # Соединяем тексты и токенизируем (убираем пустые строки)
    text = "\n\n".join(dataset["text"][:n_samples])
    encodings = tokenizer(text, return_tensors="pt")
    
    input_ids = encodings.input_ids
    nsamples = input_ids.size(1) // seq_len
    
    nll_list = [] # Сюда будем складывать Negative Log-Likelihoods
    device = next(model.parameters()).device
    
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(nsamples), desc="Computing PPL"):
            # Берем очередной кусок текста
            batch_input = input_ids[:, (i * seq_len) : ((i + 1) * seq_len)].to(device)
            target = batch_input.clone()
            
            outputs = model(batch_input, labels=target)
            
            # Hugging Face модели возвращают средний лосс по батчу/токенам в outputs.loss
            neg_log_likelihood = outputs.loss * seq_len
            nll_list.append(neg_log_likelihood)

            

    # Усредняем по всем токенам и берем экспоненту
    avg_nll = torch.stack(nll_list).sum() / (nsamples * seq_len)
    ppl = torch.exp(avg_nll)
    return ppl.item()

In [24]:
import copy
import gc
import time
import torch
from transformers import AutoModelForCausalLM

def benchmark_performance(model, input_ids, device="cuda"):
    model.to(device)
    input_ids = input_ids.to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(5):
            model(input_ids)
            
    if device == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

    # Latency
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(20):
            model(input_ids)
            
    if device == "cuda":
        torch.cuda.synchronize()
        
    latency_ms = (time.perf_counter() - t0) / 20 * 1000

    # Memory
    if device == "cuda":
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    else:
        peak_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 ** 2)
        
    return latency_ms, peak_mb


# --- Запуск бенчмарка ---

device = "cuda" if torch.cuda.is_available() else "cpu"
dummy_input = tokenizer("Hello, this is a test prompt for measuring latency.", return_tensors="pt").input_ids

# Загружаем базовую модель и оставляем её на CPU для чистого копирования
clean_model = AutoModelForCausalLM.from_pretrained("facebook/opt-125m", torch_dtype=torch.float32)

configs = [
    ("Original FP32", None),
    ("8-bit per-channel", UniformQuantizer(8, True, 'per_channel')),
    ("8-bit per-tensor", UniformQuantizer(8, True, 'per_tensor')),
    ("4-bit per-channel", UniformQuantizer(4, True, 'per_channel')),
    ("4-bit per-tensor", UniformQuantizer(4, True, 'per_tensor'))
]

print(f"{'Model Type':<20} | {'PPL':<6} | {'Latency (ms)':<12} | {'Memory (MB)':<10}")
print("-" * 57)

for name, quantizer in configs:
    # 1. Подготовка чистой копии
    model_to_test = copy.deepcopy(clean_model)
    if quantizer is not None:
        model_to_test = replace_linear_layers(model_to_test, quantizer)
    
    # 2. Метрики качества
    ppl = compute_perplexity(model_to_test, tokenizer)
    
    # 3. Метрики производительности
    lat_ms, mem_mb = benchmark_performance(model_to_test, dummy_input, device)
    
    # Вывод строки таблицы
    print(f"{name:<20} | {ppl:<6.2f} | {lat_ms:<12.2f} | {mem_mb:<10.2f}")
    
    # 4. Строгая очистка памяти перед следующим шагом
    del model_to_test
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Model Type           | PPL    | Latency (ms) | Memory (MB)
---------------------------------------------------------


Computing PPL: 100%|██████████| 23/23 [00:08<00:00,  2.65it/s]


Original FP32        | 42.19  | 60.11        | 477.75    


Computing PPL: 100%|██████████| 23/23 [00:10<00:00,  2.23it/s]


...

8-bit per-tensor     | 43.36  | 113.31       | 625.03    


Computing PPL: 100%|██████████| 23/23 [00:10<00:00,  2.14it/s]


4-bit per-channel    | 81.81  | 118.45       | 626.05    


Computing PPL: 100%|██████████| 23/23 [00:10<00:00,  2.15it/s]


4-bit per-tensor     | 7015.35 | 113.06       | 625.03    


In [29]:
def simple_gptq_row(w: torch.Tensor, X: torch.Tensor,
                    quantizer: UniformQuantizer) -> torch.Tensor:
    """
    Квантуем одну строку весов w (shape: [in_features])
    с компенсацией ошибки через информацию о входных активациях X.

    X: калибровочные активации, shape [n_samples, in_features]
    """
    # 1. Вычисляем матрицу Хессе H = X^T @ X
    #    (приблизительный гессиан loss по w)
    H = X.T @ X  # [in_features, in_features]
    H = H / X.shape[0]  # нормировка

    # 2. Добавляем damping для числовой стабильности
    H += 1e-5 * torch.eye(H.shape[0])

    # 3. Квантуем веса по одному, корректируя оставшиеся
    w_quant = w.clone()
    for i in range(len(w)):
        # Квантуем текущий вес
        w_q_i = quantizer.quantize(w_quant[i:i+1])
        w_hat_i = quantizer.dequantize(w_q_i)

        # Ошибка квантования для этого веса
        error_i = w_quant[i] - w_hat_i.squeeze()

        # Компенсируем ошибку в оставшихся весах
        # Коэффициент компенсации пропорционален H[i, i+1:]
        if i + 1 < len(w):
            compensation = error_i * H[i, i+1:] / H[i, i]
            w_quant[i+1:] -= compensation

        w_quant[i] = w_hat_i.squeeze()

    return w_quant


def simple_gptq_layer(linear: nn.Linear, calib_input: torch.Tensor,
                       quantizer: UniformQuantizer) -> torch.Tensor:
    """Оптимизированный GPTQ: один расчет Гессиана и замороженные scales."""
    W = linear.weight.data.clone()  # [out_features, in_features]
    W_q = torch.zeros_like(W)

    # 1. ЗАМОРАЖИВАЕМ SCALE И ZERO-POINT ДЛЯ ВСЕЙ МАТРИЦЫ
    # Запускаем quantize вхолостую, чтобы внутри объекта quantizer 
    # вычислились и сохранились правильные 2D-тензоры self.s и self.z
    _ = quantizer.quantize(W)
    
    # Границы сетки
    if quantizer.symmetric:
        q_max = 2**(quantizer.bits - 1) - 1
        q_min = -q_max
    else:
        q_min, q_max = 0, 2**quantizer.bits - 1

    # 2. ВЫЧИСЛЯЕМ ГЕССИАН ОДИН РАЗ НА ВЕСЬ СЛОЙ
    print("  Вычисление Гессиана...")
    H = calib_input.T @ calib_input
    H = H / calib_input.shape[0]
    H += 1e-5 * torch.eye(H.shape[0], device=H.device) # Damping

    # 3. ПОСТРОЧНЫЙ ЦИКЛ GPTQ
    for row_idx in range(W.shape[0]):
        w = W[row_idx].clone()
        
        # Достаем параметры конкретно для этой строки (канала)
        # Если per_tensor, то s и z будут скалярами, обрабатываем оба случая:
        s = quantizer.s[row_idx].item() if quantizer.s.numel() > 1 else quantizer.s.item()
        z = quantizer.z[row_idx].item() if quantizer.z.numel() > 1 else quantizer.z.item()
        
        for i in range(len(w)):
            w_i = w[i]
            
            # --- Вручную квантуем вес по заранее вычисленным параметрам ---
            w_q_i = torch.clamp(torch.round(w_i / s) + z, q_min, q_max)
            w_hat_i = (w_q_i - z) * s
            
            # Ошибка квантования
            error_i = w[i] - w_hat_i
            
            # Компенсация по Гессиану
            if i + 1 < len(w):
                compensation = error_i * H[i, i+1:] / H[i, i]
                w[i+1:] -= compensation
                
            w[i] = w_hat_i
            
        W_q[row_idx] = w

    return W_q

In [27]:
import torch
import random
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

def get_wikitext_calibration_data(tokenizer, n_samples=128, seq_len=512):
    """Загружает WikiText-2 и подготавливает 128 случайных примеров."""
    # Загружаем тестовый сплит wikitext-2
    dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    
    # Отфильтровываем пустые строки и заголовки
    texts = [text for text in dataset["text"] if len(text.strip()) > 50]
    
    # Берем случайные n_samples текстов
    random.seed(42)
    sampled_texts = random.sample(texts, n_samples)
    
    # Токенизируем
    encodings = tokenizer(
        sampled_texts, 
        truncation=True, 
        max_length=seq_len, 
        padding=True, 
        return_tensors="pt"
    )
    return encodings.input_ids

def collect_layer_inputs_gptq(model, input_ids, layer_name: str):
    """
    Пропускает данные через модель 1 раз и собирает 2D-матрицу активаций.
    Никакого backprop и обновления весов.
    """
    inputs = []
    
    def hook_fn(module, inp, out):
        # Вход inp[0] имеет форму [batch, seq_len, hidden_features]
        # Для матрицы X нам нужна плоская форма [batch * seq_len, hidden_features]
        flattened = inp[0].detach().reshape(-1, inp[0].shape[-1])
        inputs.append(flattened)

    # 1. Находим нужный слой и вешаем хук
    layer = model.get_submodule(layer_name)
    hook = layer.register_forward_hook(hook_fn)

    # 2. Строго переводим в режим инференса
    model.eval()
    input_ids = input_ids.to(model.device)
    
    # 3. Прогон без вычисления градиентов
    with torch.no_grad():
        # Для экономии VRAM можно прогонять батчами, но 128 примеров по 512 токенов
        # обычно влезают в память целиком. Если будет OOM - разбейте input_ids на чанки.
        model(input_ids)

    # 4. Снимаем хук
    hook.remove()

    # 5. Склеиваем все батчи в одну огромную матрицу X
    X = torch.cat(inputs, dim=0)
    return X


# --- ПРИМЕР ИСПОЛЬЗОВАНИЯ ---

model_id = "facebook/opt-125m"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# OPT требует pad_token, если его нет
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float32)

# 1. Получаем токены из WikiText-2
calib_tokens = get_wikitext_calibration_data(tokenizer, n_samples=128)

# 2. Выбираем слой (например, первую полносвязную матрицу первого блока)
target_layer = "model.decoder.layers.0.fc1"

# 3. Собираем активации
print(f"Сбор активаций для {target_layer}...")
X = collect_layer_inputs_gptq(model, calib_tokens, target_layer)

print(f"Форма матрицы X: {X.shape}") # Ожидается [128 * 512, 768] = [65536, 768]

# 4. Теперь можно вычислить Гессиан
H = (X.T @ X) / X.shape[0]
print(f"Форма Гессиана H: {H.shape}") # Ожидается [768, 768]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Сбор активаций для model.decoder.layers.0.fc1...
Форма матрицы X: torch.Size([65536, 768])
Форма Гессиана H: torch.Size([768, 768])


In [30]:
import copy
import gc
import torch

# 1. Создаем независимые копии модели для экспериментов
model_naive = copy.deepcopy(model)
model_gptq = copy.deepcopy(model)

# Используем 4-битное по-канальное квантование
quantizer_4b = UniformQuantizer(bits=4, symmetric=True, granularity='per_channel')

# 2. Список всех линейных слоев в первом блоке (layers.0)
target_block = "model.decoder.layers.0"
layers_to_quantize = [
    f"{target_block}.self_attn.q_proj",
    f"{target_block}.self_attn.k_proj",
    f"{target_block}.self_attn.v_proj",
    f"{target_block}.self_attn.out_proj",
    f"{target_block}.fc1",
    f"{target_block}.fc2"
]

print("--- Старт эксперимента: Квантование первого блока (INT4) ---")

for name in layers_to_quantize:
    print(f"Обработка слоя: {name} ...", end=" ")
    
    # --- НАИВНОЕ КВАНТОВАНИЕ (Round-to-Nearest) ---
    layer_naive = model_naive.get_submodule(name)
    W_orig = layer_naive.weight.data.clone()
    
    # Симулируем квантование: сжимаем в 4 бита и сразу разжимаем обратно
    W_q_naive = quantizer_4b.quantize(W_orig)
    W_hat_naive = quantizer_4b.dequantize(W_q_naive)
    
    # Если dequantize вернул лишнее измерение, убираем его
    if W_hat_naive.dim() > 2 and W_hat_naive.shape[-1] == 1:
        W_hat_naive = W_hat_naive.squeeze(-1)
        
    layer_naive.weight.data = W_hat_naive.to(W_orig.dtype)

    # --- GPTQ КВАНТОВАНИЕ ---
    layer_gptq = model_gptq.get_submodule(name)
    
    # Собираем активации X, прогоняя данные через чистую (оригинальную) модель
    X = collect_layer_inputs_gptq(model, calib_tokens, name)
    
    # Запускаем вашу функцию GPTQ
    W_gptq = simple_gptq_layer(layer_gptq, X, quantizer_4b)
    layer_gptq.weight.data = W_gptq.to(W_orig.dtype)
    
    print("Готово")
    
    # Очистка памяти видеокарты/ОЗУ после каждого слоя
    del X, W_orig, W_q_naive, W_hat_naive, W_gptq
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 3. Измерение Perplexity
print("\n--- Результаты (Perplexity) ---")
print(f"{'Original FP32:':<15}", compute_perplexity(model, tokenizer))
print(f"{'Naive INT4:':<15}", compute_perplexity(model_naive, tokenizer))
print(f"{'GPTQ INT4:':<15}", compute_perplexity(model_gptq, tokenizer))

--- Старт эксперимента: Квантование первого блока (INT4) ---
Обработка слоя: model.decoder.layers.0.self_attn.q_proj ...   Вычисление Гессиана...
Готово
Обработка слоя: model.decoder.layers.0.self_attn.k_proj ...   Вычисление Гессиана...
Готово
Обработка слоя: model.decoder.layers.0.self_attn.v_proj ...   Вычисление Гессиана...
Готово
Обработка слоя: model.decoder.layers.0.self_attn.out_proj ...   Вычисление Гессиана...
Готово
Обработка слоя: model.decoder.layers.0.fc1 ...   Вычисление Гессиана...
Готово
Обработка слоя: model.decoder.layers.0.fc2 ...   Вычисление Гессиана...
Готово

--- Результаты (Perplexity) ---


Computing PPL: 100%|██████████| 23/23 [00:11<00:00,  2.06it/s]


Original FP32:  42.187435150146484


Computing PPL: 100%|██████████| 23/23 [00:10<00:00,  2.19it/s]


Naive INT4:     44.10991287231445


Computing PPL: 100%|██████████| 23/23 [00:10<00:00,  2.21it/s]

GPTQ INT4:      4615.69482421875
